In [11]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

In [13]:
#a)

X_fantoma = np.array([
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
[0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
[0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)

x_fantoma = X_fantoma.reshape(-1)

#vuelvo a crear las matrices A2,A4,A7
def mascara_rayo(n, direccion, s):
    res =  np.zeros((n, n))
    for i in range(n) :
        for j in range(n):
            if (direccion[0]*i + direccion[1]*j) == s:
                res[i,j] =1
    return res
            

def matriz_proyeccion(n, direcciones):
    filas_A = []
    for direccion in direcciones:
        valoresdes=[]
        for i in range(n):
            for j in range(n):
                s = direccion[0]*i + direccion[1]*j
                valoresdes.append(s)
    
        ssinrepetir = []

        for k in valoresdes:
            if k not in ssinrepetir:
                ssinrepetir.append(k)        

        ssinrepetir.sort()

        for s in ssinrepetir:
            fila_s =mascara_rayo(n, direccion, s)
            filas_A.append(fila_s.reshape(-1))
           
    A=np.array(filas_A)
    return A


d7 = [(1,0), (0,1), (1,1), (1, -1), (2,1),(1,2),(3,1)]


A7= matriz_proyeccion(8, d7)

def generar_mediciones(A, x):
    return A @ x

b7 = generar_mediciones(A7, x_fantoma)


np.random.seed(42)
z = np.random.randn(len(b7))
escala = np.sqrt(np.mean(b7**2))

sigmas = [0, 0.01, 0.03, 0.06, 0.10]

b_sigma = {}
for sigma in sigmas:
    eta = sigma * escala * z
    b_sigma[sigma] = b7 + eta

In [12]:
#b)

X_fantoma = np.array([
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
[0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
[0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)
x_fantoma = X_fantoma.reshape(-1)


def mascara_rayo(n, direccion, s):
    res =  np.zeros((n, n))
    for i in range(n) :
        for j in range(n):
            if (direccion[0]*i + direccion[1]*j) == s:
                res[i,j] =1
    return res
            

def matriz_proyeccion(n, direcciones):
    filas_A = []
    for direccion in direcciones:
        valoresdes=[]
        for i in range(n):
            for j in range(n):
                s = direccion[0]*i + direccion[1]*j
                valoresdes.append(s)
    
        ssinrepetir = []

        for k in valoresdes:
            if k not in ssinrepetir:
                ssinrepetir.append(k)        

        ssinrepetir.sort()

        for s in ssinrepetir:
            fila_s =mascara_rayo(n, direccion, s)
            filas_A.append(fila_s.reshape(-1))
           
    A=np.array(filas_A)
    return A

A7= matriz_proyeccion(8, d7)

def generar_mediciones(A, x):
    return A @ x

b7 = generar_mediciones(A7, x_fantoma)   

def reconstruir(A, b):
    x_hat, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    return x_hat

np.random.seed(42)
z = np.random.randn(len(b7))
escala = np.sqrt(np.mean(b7**2))

sigmas = [0, 0.01, 0.03, 0.06, 0.10]
b_sigma = {}
for sigma in sigmas:
    eta = sigma * escala * z
    b_sigma[sigma] = b7 + eta

def residuo(Ak, bk, xk_hat, x_fantoma):
    resk = bk - Ak@xk_hat
    norma_resk = np.linalg.norm(resk)
    error_relativo = np.linalg.norm(xk_hat - x_fantoma) / np.linalg.norm(x_fantoma)
    return norma_resk, error_relativo

resultados_ruido = []
for sigma in sigmas:
    b = b_sigma[sigma]
    x_hat_sigma = reconstruir(A7, b)
    norma_ruido = np.linalg.norm(b - b7)
    norma_res, error_rel = residuo(A7, b, x_hat_sigma, x_fantoma)
    resultados_ruido.append({
        "sigma": sigma,
        "norma_ruido": norma_ruido,
        "norma_residuo": norma_res,
        "error_relativo": error_rel,
        "min_reconstruccion": x_hat_sigma.min(),
        "max_reconstruccion": x_hat_sigma.max(),
        "x_hat": x_hat_sigma
    })

tabla_ruido = pd.DataFrame(resultados_ruido)
print(tabla_ruido[["sigma", "norma_ruido", "norma_residuo", "error_relativo", "min_reconstruccion", "max_reconstruccion"]])

   sigma  norma_ruido  norma_residuo  error_relativo  min_reconstruccion  \
0   0.00     0.000000   1.986808e-14    2.550078e-15       -1.387779e-15   
1   0.01     0.110765   6.835462e-02    2.148806e-02       -1.626764e-02   
2   0.03     0.332294   2.050639e-01    6.446418e-02       -4.880291e-02   
3   0.06     0.664588   4.101277e-01    1.289284e-01       -9.760582e-02   
4   0.10     1.107647   6.835462e-01    2.148806e-01       -1.626764e-01   

   max_reconstruccion  
0            0.800000  
1            0.802564  
2            0.807692  
3            0.815384  
4            0.825641  


c) El error relativo aumenta cuando aumenta sigma, porque estamos agregando cada vez más ruido a las mediciones. Como el ruido se calcula multiplicando por sigma, cuando sigma crece también crece aproximadamente el error de la reconstrucción, por ejemplo, al multiplicar sigma por 10 (de 0.01 a 0.10), el error también se multiplica por 10 aproximadamente (de 2.15% a 21.49%). 
Cuando agregamos ruido, b_sigma ya no coincide exactamente con las mediciones de una imagen posible, por eso ⁠ lstsq ⁠ busca la mejor aproximación y queda un residuo distinto de cero.
También pueden aparecer valores negativos porque ⁠ lstsq ⁠ solo busca minimizar el error y no obliga a que los valores de los píxeles sean positivos. Por eso, con más ruido, algunos píxeles pueden tomar valores negativos aunque físicamente no tengan sentido.